--- Day 18: RAM Run ---

You and The Historians look a lot more pixelated than you remember. You're inside a computer at the North Pole!

Just as you're about to check out your surroundings, a program runs up to you. "This region of memory isn't safe! The User misunderstood what a pushdown automaton is and their algorithm is pushing whole bytes down on top of us! Run!"

The algorithm is fast - it's going to cause a byte to fall into your memory space once every nanosecond! Fortunately, you're faster, and by quickly scanning the algorithm, you create a list of which bytes will fall (your puzzle input) in the order they'll land in your memory space.

Your memory space is a two-dimensional grid with coordinates that range from 0 to 70 both horizontally and vertically. However, for the sake of example, suppose you're on a smaller grid with coordinates that range from 0 to 6 and the following list of incoming byte positions:

5,4
4,2
4,5
3,0
2,1
6,3
2,4
1,5
0,6
3,3
2,6
5,1
1,2
5,5
2,5
6,5
1,4
0,4
6,4
1,1
6,1
1,0
0,5
1,6
2,0
Each byte position is given as an X,Y coordinate, where X is the distance from the left edge of your memory space and Y is the distance from the top edge of your memory space.

You and The Historians are currently in the top left corner of the memory space (at 0,0) and need to reach the exit in the bottom right corner (at 70,70 in your memory space, but at 6,6 in this example). You'll need to simulate the falling bytes to plan out where it will be safe to run; for now, simulate just the first few bytes falling into your memory space.

As bytes fall into your memory space, they make that coordinate corrupted. Corrupted memory coordinates cannot be entered by you or The Historians, so you'll need to plan your route carefully. You also cannot leave the boundaries of the memory space; your only hope is to reach the exit.

In the above example, if you were to draw the memory space after the first 12 bytes have fallen (using . for safe and # for corrupted), it would look like this:

...#...
..#..#.
....#..
...#..#
..#..#.
.#..#..
#.#....
You can take steps up, down, left, or right. After just 12 bytes have corrupted locations in your memory space, the shortest path from the top left corner to the exit would take 22 steps. Here (marked with O) is one such path:

OO.#OOO
.O#OO#O
.OOO#OO
...#OO#
..#OO#.
.#.O#..
#.#OOOO
Simulate the first kilobyte (1024 bytes) falling onto your memory space. Afterward, what is the minimum number of steps needed to reach the exit?

--- Day 18: RAM Run ---
You and The Historians look a lot more pixelated than you remember. You're inside a computer at the North Pole!

Just as you're about to check out your surroundings, a program runs up to you. "This region of memory isn't safe! The User misunderstood what a pushdown automaton is and their algorithm is pushing whole bytes down on top of us! Run!"

The algorithm is fast - it's going to cause a byte to fall into your memory space once every nanosecond! Fortunately, you're faster, and by quickly scanning the algorithm, you create a list of which bytes will fall (your puzzle input) in the order they'll land in your memory space.

Your memory space is a two-dimensional grid with coordinates that range from 0 to 70 both horizontally and vertically. However, for the sake of example, suppose you're on a smaller grid with coordinates that range from 0 to 6 and the following list of incoming byte positions:

5,4
4,2
4,5
3,0
2,1
6,3
2,4
1,5
0,6
3,3
2,6
5,1
1,2
5,5
2,5
6,5
1,4
0,4
6,4
1,1
6,1
1,0
0,5
1,6
2,0
Each byte position is given as an X,Y coordinate, where X is the distance from the left edge of your memory space and Y is the distance from the top edge of your memory space.

You and The Historians are currently in the top left corner of the memory space (at 0,0) and need to reach the exit in the bottom right corner (at 70,70 in your memory space, but at 6,6 in this example). You'll need to simulate the falling bytes to plan out where it will be safe to run; for now, simulate just the first few bytes falling into your memory space.

As bytes fall into your memory space, they make that coordinate corrupted. Corrupted memory coordinates cannot be entered by you or The Historians, so you'll need to plan your route carefully. You also cannot leave the boundaries of the memory space; your only hope is to reach the exit.

In the above example, if you were to draw the memory space after the first 12 bytes have fallen (using . for safe and # for corrupted), it would look like this:

...#...
..#..#.
....#..
...#..#
..#..#.
.#..#..
#.#....
You can take steps up, down, left, or right. After just 12 bytes have corrupted locations in your memory space, the shortest path from the top left corner to the exit would take 22 steps. Here (marked with O) is one such path:

OO.#OOO
.O#OO#O
.OOO#OO
...#OO#
..#OO#.
.#.O#..
#.#OOOO
Simulate the first kilobyte (1024 bytes) falling onto your memory space. Afterward, what is the minimum number of steps needed to reach the exit?

Your puzzle answer was 286.

--- Part Two ---
The Historians aren't as used to moving around in this pixelated universe as you are. You're afraid they're not going to be fast enough to make it to the exit before the path is completely blocked.

To determine how fast everyone needs to go, you need to determine the first byte that will cut off the path to the exit.

In the above example, after the byte at 1,1 falls, there is still a path to the exit:

O..#OOO
O##OO#O
O#OO#OO
OOO#OO#
###OO##
.##O###
#.#OOOO
However, after adding the very next byte (at 6,1), there is no longer a path to the exit:

...#...
.##..##
.#..#..
...#..#
###..##
.##.###
#.#....
So, in this example, the coordinates of the first byte that prevents the exit from being reachable are 6,1.

Simulate more of the bytes that are about to corrupt your memory space. What are the coordinates of the first byte that will prevent the exit from being reachable from your starting position? (Provide the answer as two integers separated by a comma with no other characters.)

Your puzzle answer was 20,64.

In [1]:
from queue import Queue

import sys
sys.setrecursionlimit(10**6)

In [2]:
def read_input(file_name):
    f = []
    with open(file_name, 'r') as file:
        for row in file:
            f.append(list(row.strip().split(',')))
    return f

In [3]:
def draw_field(field):
    for r in field:
        print(''.join(r), end='\n') 

In [4]:
def encode_coordinates(p_coordinates):
    return p_coordinates[0] * 1000 + p_coordinates[1]

def decode_coordinates(p_code):
    return (p_code // 1000, p_code % 1000)

In [5]:
def initialize_field(p_max_rows, p_max_cols, p_obstacles):
    field = dict()

    field_str = ''
    for rows in range(p_max_rows):
        row = ''
        for cols in range(p_max_cols):
            field_code = encode_coordinates((rows, cols))
            #print(f'Processing field_code = {field_code} and coordinates = {(rows, cols)}')
            if [str(cols), str(rows)] not in p_obstacles:
                #print( f'[str(cols), str(rows)] = {[str(cols), str(rows)]}')
                field[field_code] = '.'
                row += '.'
            else:
                #print( f'# will be set for [str(cols), str(rows)] = {[str(cols), str(rows)]}')
                field[field_code] = '#'
                row += '#'
        field_str += row + '\n'
    field[encode_coordinates((0, 0))] = 'S'    
    field_str = 'S' + field_str[1:]
    field[encode_coordinates((rows, cols))] = 'E'
    field_str = field_str[:-2] + 'E'
    return (field, field_str, (rows, cols))

In [6]:
def fill_paths_dict(p_field, p_start, p_end, print_freq, debug=False):
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    end_paths = []

    max_x = p_end[0]
    max_y = p_end[1]

    paths_queue = Queue()
    paths_queue.put([p_start])

    itteration = 0

    if debug:
        print(f'fill_paths_dict started with itteration {itteration} and the queue has {paths_queue.qsize()}, max_x = {max_x}, max_y={max_y}')

    while not paths_queue.empty():

        itteration += 1        
        current_path = paths_queue.get()
        last_position = current_path[-1]

        if last_position == p_end and len(end_paths) == 0:
            end_paths = list(current_path)
        if last_position == p_end and len(current_path) == len(end_paths[0]):
            end_paths.append(current_path)
        elif last_position == p_end and len(current_path) < len(end_paths[0]):
            end_paths = list(current_path)

        if itteration % print_freq == 0:
            if debug:
                print(f'On itteration {itteration} the queue has {paths_queue.qsize()} paths, current_path = {current_path}, last_position = {last_position}')

        for direction in directions:
            next_position = (last_position[0] + direction[0], last_position[1] + direction[1])
            if debug:
                print(f'next_position = {next_position}')
            next_position_code = encode_coordinates(next_position)
            
            if (next_position[0] >= 0 and 
                next_position[0] <= max_x and
                next_position[1] >= 0 and 
                next_position[1] <= max_y and 
                p_field[next_position_code] != '#' and
                next_position not in current_path # avoid cycles
               ):
                    
                    next_path = list(current_path)
                    next_path.append(next_position)
                    if debug:
                        print(f'Append next_path = {next_path}')
                    paths_queue.put(next_path)
                
    return end_paths 

In [140]:
def find_path(p_path, debug=False):

    global shortest_path
    global passed_positions
    global p_field
    global print_counter

    p_position = p_path[-1]
    p_position_code = encode_coordinates(p_position)
    succsess = False

    if p_position_code not in passed_positions:
        passed_positions[p_position_code] = p_path
    elif len(p_path) >= len(passed_positions[p_position_code]):
        return False
    else:
        passed_positions[p_position_code] = p_path
    
    if p_position == (max_x, max_y):
        if len(p_path) < len(shortest_path):
            shortest_path = p_path
            return (True)
        else:
            return (False)
    
    for direction in directions:
        next_position = (p_position[0] + direction[0], p_position[1] + direction[1])
        next_position_code = encode_coordinates(next_position)
        new_path = p_path + [next_position]
       
        if (next_position[0] >= 0 and 
            next_position[0] <= max_x and
            next_position[1] >= 0 and 
            next_position[1] <= max_y and 
            p_field[next_position_code] != '#' and
            next_position not in p_path and
            len(new_path) < len(shortest_path) 
           ):
            
            if find_path(new_path):
                success = True         
    
    return succsess

In [161]:
#file_name, p_max_rows, v_bytes = 'input_example_1.txt', 7, 12
file_name, p_max_rows, v_bytes = 'input.txt', 71, 2935

p_max_cols = p_max_rows
p_obstacles = read_input(file_name)
directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
end_paths = []


(field, field_str, end_cdn) = initialize_field(p_max_rows, p_max_cols, p_obstacles[:v_bytes])
print('\nfield:')
print(field)
print('\nfield_str:')
print(field_str)

max_x = p_max_rows - 1
max_y = p_max_rows - 1

print(f'\nmax_x = {max_x}, max_y = {max_y}')

print(f'\nend_cdn = {end_cdn}')

shortest_path = [decode_coordinates(x) for x in field.keys()]
passed_positions = dict()
p_field = field
print_counter = 0

sp = find_path([(0, 0)])
print('\nshortest_path:')
print(shortest_path)
print(len(shortest_path)-1)
print(f'sp = {sp}')


field:
{0: 'S', 1: '#', 2: '.', 3: '.', 4: '.', 5: '.', 6: '.', 7: '.', 8: '.', 9: '#', 10: '.', 11: '.', 12: '.', 13: '#', 14: '#', 15: '.', 16: '.', 17: '#', 18: '.', 19: '#', 20: '.', 21: '.', 22: '.', 23: '.', 24: '.', 25: '#', 26: '.', 27: '.', 28: '.', 29: '.', 30: '#', 31: '#', 32: '.', 33: '.', 34: '.', 35: '.', 36: '.', 37: '.', 38: '.', 39: '.', 40: '.', 41: '.', 42: '.', 43: '.', 44: '.', 45: '.', 46: '.', 47: '#', 48: '#', 49: '#', 50: '.', 51: '#', 52: '#', 53: '.', 54: '.', 55: '#', 56: '.', 57: '.', 58: '.', 59: '#', 60: '.', 61: '.', 62: '.', 63: '.', 64: '.', 65: '.', 66: '.', 67: '#', 68: '#', 69: '.', 70: '.', 1000: '.', 1001: '#', 1002: '.', 1003: '#', 1004: '#', 1005: '#', 1006: '.', 1007: '#', 1008: '#', 1009: '#', 1010: '#', 1011: '#', 1012: '.', 1013: '#', 1014: '#', 1015: '#', 1016: '.', 1017: '#', 1018: '#', 1019: '#', 1020: '#', 1021: '#', 1022: '#', 1023: '#', 1024: '.', 1025: '#', 1026: '.', 1027: '#', 1028: '.', 1029: '#', 1030: '#', 1031: '#', 1032: '#',

In [163]:
sp = True
shortest_path = [decode_coordinates(x) for x in field.keys()]
initial_sp_len = len(shortest_path)
v_bytes = 2925
max_x = p_max_rows - 1
max_y = p_max_rows - 1
(field, field_str, end_cdn) = initialize_field(p_max_rows, p_max_cols, p_obstacles[:v_bytes])    
passed_positions = dict()
p_field = field
print_counter = 0    
sp = find_path([(0, 0)])

while len(shortest_path) < initial_sp_len:
    v_bytes += 1
    (field, field_str, end_cdn) = initialize_field(p_max_rows, p_max_cols, p_obstacles[:v_bytes])   
    passed_positions = dict()
    p_field = field
    print_counter = 0    
    sp = find_path([(0, 0)])
    print (f'v_bytes = {v_bytes}, p_obstacles[{v_bytes}] = {p_obstacles[v_bytes]}, len(shortest_path) = {len(shortest_path)}')   

v_bytes = 2926, p_obstacles[2926] = ['29', '8'], len(shortest_path) = 389
v_bytes = 2927, p_obstacles[2927] = ['54', '43'], len(shortest_path) = 389
v_bytes = 2928, p_obstacles[2928] = ['49', '36'], len(shortest_path) = 389
v_bytes = 2929, p_obstacles[2929] = ['24', '28'], len(shortest_path) = 389
v_bytes = 2930, p_obstacles[2930] = ['30', '7'], len(shortest_path) = 389
v_bytes = 2931, p_obstacles[2931] = ['6', '61'], len(shortest_path) = 389
v_bytes = 2932, p_obstacles[2932] = ['55', '34'], len(shortest_path) = 389
v_bytes = 2933, p_obstacles[2933] = ['9', '12'], len(shortest_path) = 389
v_bytes = 2934, p_obstacles[2934] = ['31', '40'], len(shortest_path) = 389
v_bytes = 2935, p_obstacles[2935] = ['20', '64'], len(shortest_path) = 389
v_bytes = 2936, p_obstacles[2936] = ['50', '0'], len(shortest_path) = 389
v_bytes = 2937, p_obstacles[2937] = ['66', '21'], len(shortest_path) = 389
v_bytes = 2938, p_obstacles[2938] = ['19', '18'], len(shortest_path) = 389
v_bytes = 2939, p_obstacles[29

KeyboardInterrupt: 

In [162]:
print(p_obstacles[2935])

['20', '64']
